In [23]:
import pandas as pd

df_installments = pd.read_parquet("installments.parquet")
df_previous = pd.read_parquet("previous.parquet")
df_train = pd.read_parquet("train.parquet")

df_installments = df_installments.dropna(subset=["sk_id_curr", "sk_id_prev"])
df_previous = df_previous.dropna(subset=["sk_id_curr", "sk_id_prev"])

In [ ]:
import numpy as np
import pandas as pd

installments = df_installments.copy()
train = df_train.copy()

# Diferencia entre fecha real de pago y fecha esperada
# > 0 = pagó tarde, < 0 = pagó antes
installments["payment_delay"] = (
    installments["days_entry_payment"] - installments["days_instalment"]
)

# Días de atraso: solo valores positivos
installments["dpd"] = installments["payment_delay"].clip(lower=0)

# Flag de pago tardío
installments["late_payment_flag"] = (installments["dpd"] > 0).astype(int)

# Flag de atraso severo: más de 30 días
installments["late_30_flag"] = (installments["dpd"] > 30).astype(int)

# Flag de pago menor al esperado
installments["underpaid_flag"] = (
    installments["amt_payment"] < installments["amt_instalment"]
).astype(int)

# Flag de cuota sin pago registrado
installments["missed_payment_flag"] = (
    installments["amt_payment"].fillna(0).eq(0)
).astype(int)

# Ratio pagado / esperado
installments["payment_ratio"] = (
    installments["amt_payment"] / installments["amt_instalment"]
)
installments["payment_ratio"] = installments["payment_ratio"].replace([np.inf, -np.inf], np.nan)
installments["payment_ratio"] = installments["payment_ratio"].clip(upper=3)


installments_features = installments.groupby("sk_id_curr").agg(
    # Número total de registros de cuotas/pagos del cliente
    installments_count=("num_instalment_number", "count"),

    # Número de créditos previos distintos asociados al cliente
    previous_loans_nunique=("sk_id_prev", "nunique"),

    # Mediana de días de atraso: comportamiento típico del cliente
    dpd_median=("dpd", "median"),

    # Máximo atraso observado: peor caso del cliente
    dpd_max=("dpd", "max"),

    # Proporción de pagos tardíos
    late_payment_mean=("late_payment_flag", "mean"),

    # Cantidad de pagos con más de 30 días de atraso
    late_30_sum=("late_30_flag", "sum"),

    # Total que debía pagar
    amt_instalment_sum=("amt_instalment", "sum"),

    # Total que efectivamente pagó
    amt_payment_sum=("amt_payment", "sum"),

    # Mediana del ratio pagado / esperado
    payment_ratio_median=("payment_ratio", "median"),

    # Proporción de veces que pagó menos de lo esperado
    underpaid_mean=("underpaid_flag", "mean"),

    # Proporción de cuotas sin pago registrado
    missed_payment_mean=("missed_payment_flag", "mean"),
).reset_index()

print(installments_features.head())

   sk_id_curr  installments_count  previous_loans_nunique  dpd_median  \
0      100001                   7                       2         0.0   
1      100002                  19                       1         0.0   
2      100003                  25                       3         0.0   
3      100004                   3                       1         0.0   
4      100005                   9                       1         0.0   

   dpd_max  late_payment_mean  late_30_sum  amt_instalment_sum  \
0     11.0           0.142857            0           41195.925   
1      0.0           0.000000            0          219625.695   
2      0.0           0.000000            0         1618864.650   
3      0.0           0.000000            0           21288.465   
4      1.0           0.111111            0           56161.845   

   amt_payment_sum  payment_ratio_median  underpaid_mean  missed_payment_mean  
0        41195.925                   1.0             0.0                  0.0  
1   

In [ ]:
import numpy as np
import pandas as pd

previous = df_previous.copy()
train = df_train.copy()

# Missing flags
previous["amt_annuity_missing_flag"] = previous["amt_annuity"].isna().astype("int8")
previous["amt_goods_price_missing_flag"] = previous["amt_goods_price"].isna().astype("int8")
previous["cnt_payment_missing_flag"] = previous["cnt_payment"].isna().astype("int8")

# Ratios
previous["application_credit_ratio"] = np.where(
    previous["amt_credit"] > 0,
    previous["amt_application"] / previous["amt_credit"],
    np.nan
)

previous["annuity_credit_ratio"] = np.where(
    previous["amt_credit"] > 0,
    previous["amt_annuity"] / previous["amt_credit"],
    np.nan
)

previous["goods_credit_ratio"] = np.where(
    previous["amt_credit"] > 0,
    previous["amt_goods_price"] / previous["amt_credit"],
    np.nan
)

# sellerplace_area: por su distribución rara (-1, 0, outliers), mejor separarlo así
previous["sellerplace_area_nonpositive_flag"] = previous["sellerplace_area"].fillna(0).le(0).astype("int8")
previous["sellerplace_area_positive"] = previous["sellerplace_area"].where(previous["sellerplace_area"] > 0, np.nan)

# flag_last_appl_per_contract: parece binaria por unique=2
previous["flag_last_appl_per_contract_bin"] = (
    previous["flag_last_appl_per_contract"].map({"Y": 1, "N": 0})
)


previous_num_features = previous.groupby("sk_id_curr").agg(
    # Actividad histórica
    previous_count=("sk_id_prev", "count"),

    # Recencia / antigüedad
    days_decision_max=("days_decision", "max"),        # la más reciente
    days_decision_median=("days_decision", "median"),  # antigüedad típica

    # Montos: mejor mediana por sesgo fuerte
    amt_application_median=("amt_application", "median"),
    amt_credit_median=("amt_credit", "median"),
    amt_annuity_median=("amt_annuity", "median"),
    amt_goods_price_median=("amt_goods_price", "median"),

    # Volumen total
    amt_credit_sum=("amt_credit", "sum"),

    # Plazo: mejor mediana por dispersión y missing
    cnt_payment_median=("cnt_payment", "median"),

    # Ratios: mejor mediana por estabilidad
    application_credit_ratio_median=("application_credit_ratio", "median"),
    annuity_credit_ratio_median=("annuity_credit_ratio", "median"),
    goods_credit_ratio_median=("goods_credit_ratio", "median"),

    # Missing como señal
    amt_annuity_missing_mean=("amt_annuity_missing_flag", "mean"),
    amt_goods_price_missing_mean=("amt_goods_price_missing_flag", "mean"),
    cnt_payment_missing_mean=("cnt_payment_missing_flag", "mean"),

    # Flags/proporciones
    last_appl_per_contract_mean=("flag_last_appl_per_contract_bin", "mean"),
    last_appl_in_day_mean=("nflag_last_appl_in_day", "mean"),

    # sellerplace_area tratado según su distribución
    sellerplace_area_nonpositive_mean=("sellerplace_area_nonpositive_flag", "mean"),
    sellerplace_area_positive_median=("sellerplace_area_positive", "median"),
).reset_index()


# Categóricas de baja cardinalidad
low_card_cat_cols = [
    "name_contract_type",
    "name_client_type",
    "name_product_type",
    "weekday_appr_process_start",
    "channel_type"
]

previous_cat = pd.get_dummies(
    previous[["sk_id_curr"] + low_card_cat_cols],
    columns=low_card_cat_cols,
    dummy_na=False
)

# La media de cada dummy = proporción de veces que aparece esa categoría en el historial del cliente
previous_cat_features = previous_cat.groupby("sk_id_curr").mean().reset_index()

previous_features = previous_num_features.merge(
    previous_cat_features,
    on="sk_id_curr",
    how="left"
)

print(previous_features.head())

   sk_id_curr  previous_count  days_decision_max  days_decision_median  \
0    100001.0               1            -1740.0               -1740.0   
1    100002.0               1             -606.0                -606.0   
2    100003.0               3             -746.0                -828.0   
3    100004.0               1             -815.0                -815.0   
4    100005.0               2             -315.0                -536.0   

   amt_application_median  amt_credit_median  amt_annuity_median  \
0                24835.50           23787.00            3951.000   
1               179055.00          179055.00            9251.775   
2               337500.00          348637.50           64567.665   
3                24282.00           20106.00            5357.250   
4                22308.75           20076.75            4813.200   

   amt_goods_price_median  amt_credit_sum  cnt_payment_median  ...  \
0                 24835.5         23787.0                 8.0  ...   
1     

In [ ]:
import numpy as np
import pandas as pd


def prepare_main_application(
    df: pd.DataFrame,
    one_hot: bool = True,
    drop_document_flags: bool = False,
    drop_constant_cols: bool = True,
) -> pd.DataFrame:
    """
    Prepara application_train / application_test para modelado.

    Qué hace:
    - pasa columnas a minúsculas
    - crea flags de missing útiles
    - limpia anomalías en columnas days_*
    - crea versiones en años de variables temporales
    - capea outliers en montos y conteos
    - crea versiones log1p de montos
    - crea ratios financieros y demográficos
    - agrega features de bureau, social circle, ext_source, documentos, contacto y direcciones
    - opcionalmente hace one-hot en categóricas de baja cardinalidad
    - opcionalmente elimina columnas constantes

    Nota:
    - la función solo transforma columnas que existan, así que no falla si alguna falta.
    """

    data = df.copy()
    data.columns = data.columns.str.lower()

    def safe_div(num, den):
        out = num / den
        if isinstance(out, pd.Series):
            out = out.replace([np.inf, -np.inf], np.nan)
        return out

    def cap_upper(series: pd.Series, q: float = 0.995) -> pd.Series:
        if series.dropna().empty:
            return series
        upper = series.quantile(q)
        return series.clip(upper=upper)


    # 1) Missing flags útiles
    missing_flag_cols = [
        "obs_30_cnt_social_circle",
        "obs_60_cnt_social_circle",
        "def_30_cnt_social_circle",
        "def_60_cnt_social_circle",
        "ext_source_2",
        "ext_source_3",
        "days_employed",
        "days_last_phone_change",
        "amt_annuity",
        "amt_goods_price",
        "cnt_fam_members",
        "amt_req_credit_bureau_hour",
        "amt_req_credit_bureau_day",
        "amt_req_credit_bureau_week",
        "amt_req_credit_bureau_mon",
        "amt_req_credit_bureau_qrt",
        "amt_req_credit_bureau_year",
    ]

    for col in missing_flag_cols:
        if col in data.columns:
            data[f"{col}_missing_flag"] = data[col].isna().astype("int8")


    # 2) Limpiar anomalías en days_*
    # Reglas basadas en el patrón observado:
    # - days_birth, days_registration, days_id_publish, days_last_phone_change deberían ser <= 0
    # - days_employed suele tener sentinel 365243 o positivos anómalos
    days_cols = [
        "days_birth",
        "days_employed",
        "days_registration",
        "days_id_publish",
        "days_last_phone_change",
    ]

    for col in days_cols:
        if col in data.columns:
            data[f"{col}_anom_flag"] = 0

    if "days_birth" in data.columns:
        mask = (data["days_birth"] >= 0) | (data["days_birth"] == 365243)
        data.loc[mask, "days_birth_anom_flag"] = 1
        data.loc[mask, "days_birth"] = np.nan
        data["age_years"] = -data["days_birth"] / 365

    if "days_employed" in data.columns:
        mask = (data["days_employed"] > 0) | (data["days_employed"] == 365243)
        data.loc[mask, "days_employed_anom_flag"] = 1
        data.loc[mask, "days_employed"] = np.nan
        data["employment_years"] = -data["days_employed"] / 365

    if "days_registration" in data.columns:
        mask = (data["days_registration"] > 0) | (data["days_registration"] == 365243)
        data.loc[mask, "days_registration_anom_flag"] = 1
        data.loc[mask, "days_registration"] = np.nan
        data["registration_years"] = -data["days_registration"] / 365

    if "days_id_publish" in data.columns:
        mask = (data["days_id_publish"] > 0) | (data["days_id_publish"] == 365243)
        data.loc[mask, "days_id_publish_anom_flag"] = 1
        data.loc[mask, "days_id_publish"] = np.nan
        data["id_publish_years"] = -data["days_id_publish"] / 365

    if "days_last_phone_change" in data.columns:
        mask = (data["days_last_phone_change"] > 0) | (data["days_last_phone_change"] == 365243)
        data.loc[mask, "days_last_phone_change_anom_flag"] = 1
        data.loc[mask, "days_last_phone_change"] = np.nan
        data["last_phone_change_years"] = -data["days_last_phone_change"] / 365


    # 3) Montos: cap + log
    amount_cols = [
        "amt_income_total",
        "amt_credit",
        "amt_annuity",
        "amt_goods_price",
    ]

    for col in amount_cols:
        if col in data.columns:
            data[f"{col}_cap"] = cap_upper(data[col], q=0.995)
            data[f"{col}_log"] = np.log1p(data[f"{col}_cap"])


    # 4) Conteos: cap
    count_like_cols = [
        "cnt_children",
        "cnt_fam_members",
        "obs_30_cnt_social_circle",
        "def_30_cnt_social_circle",
        "obs_60_cnt_social_circle",
        "def_60_cnt_social_circle",
        "amt_req_credit_bureau_hour",
        "amt_req_credit_bureau_day",
        "amt_req_credit_bureau_week",
        "amt_req_credit_bureau_mon",
        "amt_req_credit_bureau_qrt",
        "amt_req_credit_bureau_year",
    ]

    for col in count_like_cols:
        if col in data.columns:
            data[f"{col}_cap"] = cap_upper(data[col], q=0.99)


    # 5) Ratios financieros y demográficos
    if {"amt_credit", "amt_income_total"}.issubset(data.columns):
        data["credit_income_ratio"] = safe_div(data["amt_credit"], data["amt_income_total"] + 1)

    if {"amt_annuity", "amt_income_total"}.issubset(data.columns):
        data["annuity_income_ratio"] = safe_div(data["amt_annuity"], data["amt_income_total"] + 1)

    if {"amt_annuity", "amt_credit"}.issubset(data.columns):
        data["annuity_credit_ratio"] = safe_div(data["amt_annuity"], data["amt_credit"] + 1)

    if {"amt_goods_price", "amt_credit"}.issubset(data.columns):
        data["goods_credit_ratio"] = safe_div(data["amt_goods_price"], data["amt_credit"] + 1)

    if {"amt_income_total", "cnt_fam_members"}.issubset(data.columns):
        data["income_per_family_member"] = safe_div(
            data["amt_income_total"], data["cnt_fam_members"].fillna(1) + 1
        )

    if {"amt_income_total", "cnt_children"}.issubset(data.columns):
        data["income_per_child"] = safe_div(data["amt_income_total"], data["cnt_children"] + 1)

    if {"employment_years", "age_years"}.issubset(data.columns):
        data["employment_age_ratio"] = safe_div(data["employment_years"], data["age_years"] + 1e-6)

    if {"cnt_children", "cnt_fam_members"}.issubset(data.columns):
        data["children_family_ratio"] = safe_div(
            data["cnt_children"], data["cnt_fam_members"].fillna(1) + 1e-6
        )

    # cap a ratios para evitar extremos
    ratio_cols = [
        "credit_income_ratio",
        "annuity_income_ratio",
        "annuity_credit_ratio",
        "goods_credit_ratio",
        "income_per_family_member",
        "income_per_child",
        "employment_age_ratio",
        "children_family_ratio",
    ]

    for col in ratio_cols:
        if col in data.columns:
            data[col] = data[col].replace([np.inf, -np.inf], np.nan)
            data[col] = cap_upper(data[col], q=0.995)


    # 6) EXT_SOURCE
    ext_cols = [c for c in ["ext_source_1", "ext_source_2", "ext_source_3"] if c in data.columns]
    if len(ext_cols) >= 2:
        data["ext_sources_mean"] = data[ext_cols].mean(axis=1)
        data["ext_sources_min"] = data[ext_cols].min(axis=1)
        data["ext_sources_max"] = data[ext_cols].max(axis=1)
        data["ext_sources_std"] = data[ext_cols].std(axis=1)


    # 7) Bureau requests
    bureau_cols = [
        c for c in [
            "amt_req_credit_bureau_hour",
            "amt_req_credit_bureau_day",
            "amt_req_credit_bureau_week",
            "amt_req_credit_bureau_mon",
            "amt_req_credit_bureau_qrt",
            "amt_req_credit_bureau_year",
        ] if c in data.columns
    ]

    if bureau_cols:
        data["bureau_requests_total"] = data[bureau_cols].fillna(0).sum(axis=1)

    recent_bureau_cols = [
        c for c in [
            "amt_req_credit_bureau_hour",
            "amt_req_credit_bureau_day",
            "amt_req_credit_bureau_week",
            "amt_req_credit_bureau_mon",
        ] if c in data.columns
    ]
    if recent_bureau_cols:
        data["bureau_requests_recent"] = data[recent_bureau_cols].fillna(0).sum(axis=1)


    # 8) Social circle
    obs_cols = [c for c in ["obs_30_cnt_social_circle", "obs_60_cnt_social_circle"] if c in data.columns]
    def_cols = [c for c in ["def_30_cnt_social_circle", "def_60_cnt_social_circle"] if c in data.columns]

    if obs_cols:
        data["social_obs_total"] = data[obs_cols].fillna(0).sum(axis=1)

    if def_cols:
        data["social_def_total"] = data[def_cols].fillna(0).sum(axis=1)

    if {"social_obs_total", "social_def_total"}.issubset(data.columns):
        data["social_def_ratio"] = safe_div(data["social_def_total"], data["social_obs_total"] + 1)
        data["social_def_ratio"] = cap_upper(data["social_def_ratio"], q=0.995)


    # 9) Contactabilidad
    contact_cols = [c for c in ["flag_emp_phone", "flag_work_phone", "flag_phone", "flag_email"] if c in data.columns]
    if contact_cols:
        data["num_contact_channels"] = data[contact_cols].fillna(0).clip(0, 1).sum(axis=1)


    # 10) Inconsistencia de direcciones
    addr_cols = [
        c for c in [
            "reg_region_not_live_region",
            "reg_region_not_work_region",
            "live_region_not_work_region",
            "reg_city_not_live_city",
            "reg_city_not_work_city",
            "live_city_not_work_city",
        ] if c in data.columns
    ]
    if addr_cols:
        data["address_mismatch_count"] = data[addr_cols].fillna(0).clip(0, 1).sum(axis=1)

  
    # 11) Documentos
    doc_cols = [c for c in data.columns if c.startswith("flag_document_")]
    if doc_cols:
        data["num_documents_provided"] = data[doc_cols].fillna(0).clip(0, 1).sum(axis=1)


    # 12) One-hot en categóricas de baja cardinalidad
    low_card_cat_cols = [
        "name_contract_type",
        "code_gender",
        "flag_own_car",
        "flag_own_realty",
        "name_income_type",
        "name_education_type",
        "name_family_status",
        "name_housing_type",
        "weekday_appr_process_start",
    ]
    low_card_cat_cols = [c for c in low_card_cat_cols if c in data.columns]

    if one_hot and low_card_cat_cols:
        data = pd.get_dummies(data, columns=low_card_cat_cols, dummy_na=True)


    # 13) Quitar flags documentales crudos
    if drop_document_flags and doc_cols:
        data = data.drop(columns=doc_cols)


    # 14) Quitar columnas constantes
    if drop_constant_cols:
        nunique = data.nunique(dropna=False)
        constant_cols = nunique[nunique <= 1].index.tolist()
        if constant_cols:
            data = data.drop(columns=constant_cols)

    return data

In [ ]:
train_prepared = prepare_main_application(df_train, one_hot=True)
print(train_prepared.head())

   sk_id_curr  cnt_children  amt_income_total  amt_credit  amt_annuity  \
0      100002             0          202500.0    406597.5      24700.5   
1      100031             0          112500.0    979992.0      27076.5   
2      100047             0          202500.0   1193580.0      35028.0   
3      100049             0          135000.0    288873.0      16258.5   
4      100096             0           81000.0    252000.0      14593.5   

   amt_goods_price name_type_suite  region_population_relative  days_birth  \
0         351000.0   Unaccompanied                    0.018801     -9461.0   
1         702000.0   Unaccompanied                    0.018029    -18724.0   
2         855000.0   Unaccompanied                    0.025164    -17482.0   
3         238500.0   Unaccompanied                    0.007305    -13384.0   
4         252000.0   Unaccompanied                    0.028663    -24794.0   

   days_employed  ...  weekday_appr_process_start_1  \
0         -637.0  ...          